# Portfolio Value Over Time

This notebook combines transaction dates and trading-price dates. For each date, it loads the latest position available on or before that date, values each position using the latest available trading price, and plots the total portfolio value.

In [ ]:
from datetime import date
from decimal import Decimal
from pathlib import Path
import sys

project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root))

from src.core.repositories.position_repository import PositionRepository
from src.core.repositories.trading_price_repository import TradingPriceRepository
from src.core.repositories.transaction_repository import TransactionRepository
from src.core.services.position_value_service import PositionValueService

In [ ]:
position_repository = PositionRepository()
trading_price_repository = TradingPriceRepository()
transaction_repository = TransactionRepository()
position_value_service = PositionValueService(trading_price_repository)

transaction_dates = {transaction.date for transaction in transaction_repository.all()}
trading_price_dates = {
    trading_price.evaluation_date
    for trading_price in trading_price_repository.all()
}
evaluation_dates = sorted(transaction_dates | trading_price_dates)

print(f'Transaction dates: {len(transaction_dates)}')
print(f'Trading-price dates: {len(trading_price_dates)}')
print(f'Evaluation dates: {len(evaluation_dates)}')

In [ ]:
portfolio_values = []
for evaluation_date in evaluation_dates:
    positions = position_repository.all(evaluation_date)
    portfolio_value = sum(
        (
            position_value_service.value(position, evaluation_date).value
            for position in positions
        ),
        Decimal('0'),
    )
    portfolio_values.append({
        'date': evaluation_date,
        'portfolio_value': portfolio_value,
        'positions': len(positions),
    })

import pandas as pd
from IPython.display import display

portfolio_value_table = pd.DataFrame(portfolio_values)
display(portfolio_value_table.tail())

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14, 6))
plt.plot(
    portfolio_value_table['date'],
    portfolio_value_table['portfolio_value'].astype(float),
    linewidth=2,
)
plt.title('Portfolio Value Over Time')
plt.xlabel('Date')
plt.ylabel('Portfolio value')
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()